In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re
import time
from html import unescape
from collections import Counter

In [2]:
CATEGORY_MAP = {
    "economics": [
        "econ.EM",   # Econometrics
        "econ.GN",   # General Economics
        "econ.TH"    # Theoretical Economics
    ],
    "electrical_engineering": [
        "eess.AS",   # Audio and Speech Processing
        "eess.IV",   # Image and Video Processing
        "eess.SP",   # Signal Processing
        "eess.SY"    # Systems and Control
    ],
    "biology": [
        "q-bio.BM",  # Biomolecules
        "q-bio.GN",  # Genomics
        "q-bio.NC"   # Neurons and Cognition
    ]
}

In [3]:
def flatten_category_map(category_map):
    rows = []
    for broad_subject, subcats in category_map.items():
        for subcat in subcats:
            rows.append((broad_subject, subcat))
    return rows

flat_categories = flatten_category_map(CATEGORY_MAP)
flat_categories

[('economics', 'econ.EM'),
 ('economics', 'econ.GN'),
 ('economics', 'econ.TH'),
 ('electrical_engineering', 'eess.AS'),
 ('electrical_engineering', 'eess.IV'),
 ('electrical_engineering', 'eess.SP'),
 ('electrical_engineering', 'eess.SY'),
 ('biology', 'q-bio.BM'),
 ('biology', 'q-bio.GN'),
 ('biology', 'q-bio.NC')]

In [4]:
def fetch_primary_category_papers(category, target_primary_count=2000, batch_size=500, sort_by="submittedDate"):
    all_rows = []
    start = 0

    BASE_URL = "http://export.arxiv.org/api/query"

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    stop_reason = "unknown"

    while len(all_rows) < target_primary_count and start < 30000:
        current_batch = min(batch_size, 2000)

        params = {
            "search_query": f"cat:{category}",
            "start": start,
            "max_results": current_batch,
            "sortBy": sort_by,
            "sortOrder": "descending"
        }

        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()

        root = ET.fromstring(response.text)
        entries = root.findall("atom:entry", ns)

        if not entries:
            stop_reason = "no more entries returned"
            break

        kept_this_batch = 0

        for entry in entries:
            title = entry.findtext("atom:title", default="", namespaces=ns)
            summary = entry.findtext("atom:summary", default="", namespaces=ns)
            published = entry.findtext("atom:published", default="", namespaces=ns)
            updated = entry.findtext("atom:updated", default="", namespaces=ns)
            paper_id = entry.findtext("atom:id", default="", namespaces=ns)

            authors = entry.findall("atom:author", ns)
            author_names = [a.findtext("atom:name", default="", namespaces=ns) for a in authors]

            categories_xml = entry.findall("atom:category", ns)
            category_terms = [c.attrib.get("term", "") for c in categories_xml]

            primary_category = ""
            primary_cat_elem = entry.find("arxiv:primary_category", ns)
            if primary_cat_elem is not None:
                primary_category = primary_cat_elem.attrib.get("term", "")

            if primary_category != category:
                continue

            row = {
                "id": paper_id,
                "title": title.strip(),
                "abstract": summary.strip(),
                "published": published,
                "updated": updated,
                "authors": ", ".join(author_names),
                "primary_category": primary_category,
                "all_categories": ", ".join(category_terms),
                "queried_category": category
            }

            all_rows.append(row)
            kept_this_batch += 1

            if len(all_rows) >= target_primary_count:
                stop_reason = "target reached"
                break

        print(
            f"{category}: start={start}, batch_returned={len(entries)}, "
            f"kept_primary={kept_this_batch}, total_kept={len(all_rows)}"
        )

        start += current_batch
        time.sleep(3)

    if start >= 30000 and len(all_rows) < target_primary_count:
        stop_reason = "hit start limit"

    print(f"{category}: STOPPED because -> {stop_reason}")
    return pd.DataFrame(all_rows)

In [5]:
TARGET_PER_PRIMARY_CATEGORY = 2000

all_dfs = []

for broad_subject, subcat in flat_categories:
    print(f"\nFetching primary papers for {subcat} under {broad_subject}...")
    
    temp_df = fetch_primary_category_papers(
        category=subcat,
        target_primary_count=TARGET_PER_PRIMARY_CATEGORY,
        batch_size=1000
    )
    
    temp_df["broad_subject"] = broad_subject
    all_dfs.append(temp_df)

df_raw = pd.concat(all_dfs, ignore_index=True)

print("\nTotal rows collected:", len(df_raw))


Fetching primary papers for econ.EM under economics...
econ.EM: start=0, batch_returned=1000, kept_primary=694, total_kept=694
econ.EM: start=1000, batch_returned=1000, kept_primary=728, total_kept=1422
econ.EM: start=2000, batch_returned=1000, kept_primary=578, total_kept=2000
econ.EM: STOPPED because -> target reached

Fetching primary papers for econ.GN under economics...
econ.GN: start=0, batch_returned=1000, kept_primary=750, total_kept=750
econ.GN: start=1000, batch_returned=1000, kept_primary=770, total_kept=1520
econ.GN: start=2000, batch_returned=1000, kept_primary=480, total_kept=2000
econ.GN: STOPPED because -> target reached

Fetching primary papers for econ.TH under economics...
econ.TH: start=0, batch_returned=1000, kept_primary=746, total_kept=746
econ.TH: start=1000, batch_returned=1000, kept_primary=718, total_kept=1464
econ.TH: start=2000, batch_returned=1000, kept_primary=536, total_kept=2000
econ.TH: STOPPED because -> target reached

Fetching primary papers for ee

In [6]:
print(df_raw["primary_category"].value_counts())

primary_category
econ.EM     2000
econ.GN     2000
econ.TH     2000
eess.AS     2000
eess.IV     2000
eess.SP     2000
eess.SY     2000
q-bio.BM    2000
q-bio.GN    2000
q-bio.NC    2000
Name: count, dtype: int64


In [7]:
df_raw.to_csv("data/arxiv_raw_metadata.csv", index=False)
df_raw.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject
0,http://arxiv.org/abs/2603.20134v1,Triple/Double-Debiased Lasso,"In this paper, we propose a triple (or double-...",2026-03-20T17:07:09Z,2026-03-20T17:07:09Z,"Denis Chetverikov, Jesper R. -V. Sørensen, Ale...",econ.EM,"econ.EM, math.ST",econ.EM,economics
1,http://arxiv.org/abs/2603.18870v1,Inference in Regression Discontinuity Designs ...,Clustered sampling is prevalent in empirical r...,2026-03-19T13:15:49Z,2026-03-19T13:15:49Z,"Claudia Noack, Tomasz Olma, Christoph Rothe",econ.EM,"econ.EM, stat.ME",econ.EM,economics
2,http://arxiv.org/abs/2603.17881v1,Towards Measuring Disruptive Innovation Across...,The CD index is a widely used measure of disru...,2026-03-18T16:05:21Z,2026-03-18T16:05:21Z,"Christian Rutzer, Dragan Filimonovic, Jeffrey ...",econ.EM,econ.EM,econ.EM,economics
3,http://arxiv.org/abs/2603.17381v3,An Auditable AI Agent Loop for Empirical Econo...,AI coding agents make empirical specification ...,2026-03-18T05:55:04Z,2026-03-20T06:29:04Z,Minchul Shin,econ.EM,"econ.EM, stat.ML",econ.EM,economics
4,http://arxiv.org/abs/2603.16035v1,Identification Verification for Structural Vec...,We propose a structural vector autoregressive ...,2026-03-17T00:41:19Z,2026-03-17T00:41:19Z,"Fei Shang, Tomasz Woźniak",econ.EM,econ.EM,econ.EM,economics


In [8]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = unescape(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_cleaning(df):
    df = df.copy()

    df["title_clean"] = df["title"].apply(clean_text)
    df["abstract_clean"] = df["abstract"].apply(clean_text)
    df["text"] = df["title_clean"] + " " + df["abstract_clean"]

    df["abstract_word_count"] = df["abstract_clean"].apply(lambda x: len(x.split()))
    df["title_word_count"] = df["title_clean"].apply(lambda x: len(x.split()))

    # remove duplicates by id
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    # remove very short abstracts
    df = df[df["abstract_word_count"] >= 20].reset_index(drop=True)

    return df

df = simple_cleaning(df_raw)
print("Rows b4 cleaning:", len(df_raw))
print("Rows after cleaning:", len(df))

Rows b4 cleaning: 20000
Rows after cleaning: 19988


In [9]:
# --- handle arXiv versions properly ---

df["base_id"] = df["id"].str.extract(r"(\d+\.\d+)")
df["version"] = df["id"].str.extract(r"v(\d+)$").astype(float)

print("Before dedup:", len(df))

df = df.sort_values(by=["base_id", "version"], ascending=[True, False])
df = df.drop_duplicates(subset="base_id", keep="first").reset_index(drop=True)

print("After dedup:", len(df))

Before dedup: 19988
After dedup: 19974


In [10]:
# compute base_id and version, count versions per paper
df['base_id'] = df['id'].str.extract(r'(\d+\.\d+)(?:v\d+)?$')[0]
df['version'] = df['id'].str.extract(r'v(\d+)$')[0].fillna('1').astype(int)

# count how many versions are present per base_id
version_counts = df.groupby('base_id').size().rename('n_versions')
print('Number of distinct base papers:', version_counts.shape[0])
print('How many base papers have multiple versions:', (version_counts>1).sum())
print('\nDistribution of version counts (n_versions: #base papers):')
print(version_counts.value_counts().sort_index())

# show a few examples of papers with multiple versions
multi = version_counts[version_counts>1].index
if len(multi):
    print('\nExample entries for papers with multiple versions:')
    display(df[df['base_id'].isin(multi)].sort_values(['base_id','version']).head(20))
else:
    print('\nNo multi-version papers found in current dataframe.')

# create a dataframe with latest version per base_id
df_latest = (
    df.sort_values(['base_id','version'], ascending=[True,False])
      .drop_duplicates(subset='base_id', keep='first')
      .reset_index(drop=True)
)
print('\nOriginal rows:', len(df))
print('Rows after keeping latest version only:', len(df_latest))

# assign back if you want to use latest-only going forward:
# df = df_latest.copy()


Number of distinct base papers: 19973
How many base papers have multiple versions: 0

Distribution of version counts (n_versions: #base papers):
n_versions
1    19973
Name: count, dtype: int64

No multi-version papers found in current dataframe.

Original rows: 19974
Rows after keeping latest version only: 19974


In [11]:
df.to_csv("data/arxiv_cleaned_metadata.csv", index=False)
df.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject,title_clean,abstract_clean,text,abstract_word_count,title_word_count,base_id,version
0,http://arxiv.org/abs/0704.1169v1,Holographic bound and protein linguistics,The holographic bound in physics constrains th...,2007-04-10T12:04:28Z,2007-04-10T12:04:28Z,"Dirson Jian Li, Shengli Zhang",q-bio.GN,"q-bio.GN, hep-th, q-bio.QM",q-bio.GN,biology,holographic bound and protein linguistics,the holographic bound in physics constrains th...,holographic bound and protein linguistics the ...,66,5,0704.1169,1
1,http://arxiv.org/abs/0704.2191v2,Mismatch Repair Error Implies Chargaff's Secon...,Chargaff's second parity rule holds empiricall...,2007-04-17T16:15:37Z,2007-09-20T15:46:34Z,Bo Deng,q-bio.GN,q-bio.GN,q-bio.GN,biology,mismatch repair error implies chargaff's secon...,chargaff's second parity rule holds empiricall...,mismatch repair error implies chargaff's secon...,155,8,0704.2191,2
2,http://arxiv.org/abs/0704.3826v2,Non-coding DNA programs express adaptation and...,Significant fraction (98.5% in humans) of most...,2007-04-29T05:32:50Z,2007-08-02T16:23:54Z,Mark Ya. Azbel,q-bio.GN,"q-bio.GN, cond-mat.other, nlin.AO, q-bio.OT, q...",q-bio.GN,biology,non-coding dna programs express adaptation and...,significant fraction (98.5% in humans) of most...,non-coding dna programs express adaptation and...,161,9,0704.3826,2
3,http://arxiv.org/abs/0705.1656v1,Chromatin Folding in Relation to Human Genome ...,Three-dimensional (3D) chromatin structure is ...,2007-05-11T14:12:05Z,2007-05-11T14:12:05Z,"Julio Mateos-Langerak, Osdilly Giromus, Wim de...",q-bio.GN,q-bio.GN,q-bio.GN,biology,chromatin folding in relation to human genome ...,three-dimensional (3d) chromatin structure is ...,chromatin folding in relation to human genome ...,226,8,0705.1656,1
4,http://arxiv.org/abs/0705.2143v1,Inverted and mirror repeats in model nucleotid...,We analytically and numerically study the prob...,2007-05-15T13:37:17Z,2007-05-15T13:37:17Z,"Fabrizio Lillo, Marco Spanó",q-bio.GN,"q-bio.GN, q-bio.QM",q-bio.GN,biology,inverted and mirror repeats in model nucleotid...,we analytically and numerically study the prob...,inverted and mirror repeats in model nucleotid...,79,8,0705.2143,1


In [12]:
print(df["abstract_word_count"].describe())

count    19974.000000
mean       165.864173
std         50.664573
min         20.000000
25%        130.000000
50%        162.000000
75%        200.000000
max        539.000000
Name: abstract_word_count, dtype: float64


In [13]:
df[["title", "primary_category", "published"]].sample(10, random_state=42)

,title,primary_category,published
6248,Quantifying Signal-to-Noise Ratio in Neural La...,q-bio.NC,2024-08-16T13:53:52Z
15199,Strategic bid response under automated market ...,econ.GN,2025-11-25T19:57:20Z
18927,Spatial Degrees of Freedom in Near Field MIMO:...,eess.SP,2026-02-25T14:29:26Z
15013,State-of-charge estimation of lithium-ion batt...,eess.SP,2025-11-21T02:05:44Z
16573,Geometry-Aware Optimization for Respiratory So...,eess.AS,2025-12-27T11:39:36Z
11791,The Role of High-Speed Rail in Reshaping Chine...,econ.GN,2025-08-22T04:13:29Z
9072,Impact of random monetary shock: a Keynesian case,econ.TH,2025-05-01T19:01:13Z
9390,NOVA: A Benchmark for Anomaly Localization and...,eess.IV,2025-05-20T08:10:57Z
10210,Breaking the Multi-Enhancement Bottleneck: Dom...,eess.IV,2025-06-17T03:30:33Z
9226,A portable diagnosis model for Keratoconus usi...,eess.IV,2025-05-13T14:34:46Z


In [33]:
print(df.loc[1, 'title'])

Mismatch Repair Error Implies Chargaff's Second Parity Rule


In [ ]:
# Data pre-processing

In [34]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

stop_list = stopwords.words('english')
stemmer = PorterStemmer()

def preprocess(doc):
    words = doc.lower().split()
    words = [w for w in words if re.search('^[a-z]+$', w)]
    words = [w for w in words if w not in stop_list]
    words = [stemmer.stem(w) for w in words]
    return words

In [15]:
##Basic Naive Bayes Model - no validation data

In [35]:
df['text'] = df['title_clean'] + ' ' + df['abstract_clean']
df['label'] = df['primary_category']   

import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

df['tokens'] = df['text'].apply(preprocess)

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

from gensim import corpora

dictionary = corpora.Dictionary(train_df['tokens'])

def doc_to_features(tokens):
    bow = dictionary.doc2bow(tokens)
    return {id:1 for (id, tf_value) in bow}

train_features = [doc_to_features(doc) for doc in train_df['tokens']]
test_features = [doc_to_features(doc) for doc in test_df['tokens']]

train_data = list(zip(train_features, train_df['label']))
test_data = list(zip(test_features, test_df['label']))

import nltk

classifier = nltk.NaiveBayesClassifier.train(train_data)

print("Accuracy:", round(nltk.classify.accuracy(classifier, test_data) * 100,2))

Accuracy: 83.75


In [ ]:
#Check whether model can predict

def predict_new_paper(title, abstract, classifier, dictionary):
    text = title + " " + abstract
    tokens = preprocess(text)                  # reuse the same preprocess function
    features = {id:1 for (id, tf_value) in dictionary.doc2bow(tokens)}
    return classifier.classify(features)

# Example usage
sample_title = "Mismatch Repair Error Implies Chargaff's Second Parity Rule"
sample_abstract = "Chargaff's second parity rule holds empirically for most types of DNA that along single strands of DNA the base contents are equal for complimentary bases, A = T, G = C. A Markov chain model is constructed to track the evolution of any single base position along single strands of genomes whose organisms are equipped with replication mismatch repair. Under the key assumptions that mismatch error rates primarily depend the number of hydrogen bonds of nucleotides and that the mismatch repairing process itself makes strand recognition error, the model shows that the steady state probabilities for any base position to take on one of the 4 nucleotide bases are equal for complimentary bases. As a result, Chargaff's second parity rule is the manifestation of the Law of Large Number acting on the steady state probabilities. More importantly, because the model pinpoints mismatch repair as a basis of the rule, it is suitable for experimental verification."
predicted_category = predict_new_paper(sample_title, sample_abstract, classifier, dictionary)
print("Predicted Category:", predicted_category)

Predicted Category: q-bio.GN


In [17]:
#Naive Bayes With Feature Engineering - no validation data

In [18]:
df['title_tokens'] = df['title_clean'].apply(preprocess)
df['abstract_tokens'] = df['abstract_clean'].apply(preprocess)

from gensim import corpora

all_tokens = df['title_tokens'] + df['abstract_tokens']
dictionary = corpora.Dictionary(all_tokens)

def doc_to_features(title_tokens, abstract_tokens):
    features = {}

    # Title words (more important)
    for word in title_tokens:
        features[f"title_{word}"] = 1

    # Abstract words
    for word in abstract_tokens:
        features[f"abstract_{word}"] = 1

    # length features 
    # number of words in title
    # number of words in abstract (Longer abstracts may belong to certain categories)
    features["title_length"] = len(title_tokens)
    features["abstract_length"] = len(abstract_tokens)

    return features

# Where the word came from (title vs abstract)
df['features'] = df.apply(
    lambda row: doc_to_features(row['title_tokens'], row['abstract_tokens']),
    axis=1
)


from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_data = list(zip(train_df['features'], train_df['label']))
test_data = list(zip(test_df['features'], test_df['label']))

import nltk
classifier = nltk.NaiveBayesClassifier.train(train_data)

print("Accuracy:", round(nltk.classify.accuracy(classifier, test_data) * 100,2))

Accuracy: 85.41


In [53]:
# Function to predict new paper based on feature-engineered model
def predict_new_paper_feature_engineered(title, abstract, classifier):
    # Preprocess title and abstract
    title_tokens = preprocess(title)
    abstract_tokens = preprocess(abstract)
    
    # Create features the same way as training
    features = {}
    
    # Title words
    for word in title_tokens:
        features[f"title_{word}"] = 1
    
    # Abstract words
    for word in abstract_tokens:
        features[f"abstract_{word}"] = 1
    
    # Length features
    features["title_length"] = len(title_tokens)
    features["abstract_length"] = len(abstract_tokens)
    
    # Predict category
    predicted_label = classifier.classify(features)
    
    return predicted_label

sample_title = "Robust Linear Quadratic Optimal Control of Cementitious Material Extrusion"
sample_abstract = "Extrusion-based 3D printing of cementitious materials enables fabrication of complex structures, however it is highly sensitive to disturbances, material property variations, and process uncertainties that decrease flow stability and dimensional fidelity. To address these challenges, this study proposes a robust linear quadratic optimal control framework for regulating material extrusion in cementitious direct ink writing systems. The printer is modeled using two coupled subsystems: an actuation system representing nozzle flow dynamics and a printing system describing the printed strand flow on the build plate. A hybrid control architecture combining sliding mode control for disturbance rejection with linear quadratic optimal feedback for energy-efficient tracking is developed to ensure robustness and optimality. In simulation case studies, the control architecture guarantees acceptable convergence of nozzle and strand flow tracking errors under bounded disturbances."

predicted_category = predict_new_paper_feature_engineered(
    sample_title,
    sample_abstract,
    classifier
)

print("Predicted Category:", predicted_category)

Predicted Category: econ.EM


In [51]:

print(df.loc[19972, 'title'])


Robust Linear Quadratic Optimal Control of Cementitious Material Extrusion


In [54]:
train_df['label'].value_counts()

label
econ.EM     1637
q-bio.GN    1633
eess.SY     1624
q-bio.BM    1594
econ.GN     1591
econ.TH     1586
eess.IV     1585
eess.SP     1580
q-bio.NC    1579
eess.AS     1570
Name: count, dtype: int64